# Chapter 15 – Recurrences: RecursiveSum, Tower of Hanoi and closed forms (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch15_recurrences_cpp.ipynb)

Numerical check of the examples of the chapter: the algorithm **RecursiveSum** with the count of the steps $S(n)$ (which satisfies $S(1)=2$, $S(n)=S(n-1)+3$, that is $S(n)=3n-1$), the **Tower of Hanoi**, which counts the moves ($2^n-1$), and the comparison between some recurrences of the chapter, computed directly, and their closed form.

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/analysis-recursion/15-recurrences/).

Each program is written to a file by the `%%writefile` cell and then compiled and run with `g++ -std=c++17 -O2`: run the cells in order, from top to bottom. No external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `RecursiveSum` (PascalCase).

## 1. RecursiveSum and the count of the steps

**RecursiveSum**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the sum $a[1]+\dots+a[n]$.

The code follows the pseudo-code (lines 2, 3, 5, 6); a counter `steps` adds up the steps executed by every call, as in the chapter: 2 steps if $n=1$, 3 steps plus the recursive call if $n>1$. The second function is `sumRecDecrease` of the code of the course, which stops at the empty array ($n=0$) and therefore makes one more call.

In [1]:
%%writefile recursive_sum.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long steps = 0;   // steps executed (lines of pseudo-code)
int calls = 0;         // number of calls

// Sum of a[0..n-1] (pseudo-code: a[1..n]), n >= 1
int RecursiveSum(int *a, int n)
{
    calls++;
    steps++;                  // line 2: if n = 1
    if (n == 1)
    {
        steps++;              // line 3: return a[1]
        return a[0];
    }
    steps += 2;               // lines 5 and 6
    int s = RecursiveSum(a, n - 1);
    return s + a[n - 1];
}

// Variant of the course code: base case n = 0
long long sumRecDecrease(int *a, int n)
{
    calls++;
    if (n == 0)
    {
        return 0;
    }
    return sumRecDecrease(a, n - 1) + a[n - 1];
}

#ifndef NOMAIN
int main()
{
    int a[] = {3, 1, 4, 1, 5, 9, 2, 6};
    int n = 8;
    printf("a = [3, 1, 4, 1, 5, 9, 2, 6]\n");
    calls = 0;
    int r = RecursiveSum(a, n);
    printf("RecursiveSum(a, 8) = %d with %d calls\n", r, calls);
    calls = 0;
    long long r2 = sumRecDecrease(a, n);
    printf("sumRecDecrease(a, 8) = %lld with %d calls\n", r2, calls);
    printf("n  measured S(n)  3n-1\n");
    for (int k = 1; k <= 8; k++)
    {
        steps = 0;
        RecursiveSum(a, k);
        printf("%-2d %12lld %5d\n", k, steps, 3 * k - 1);
    }
    return 0;
}
#endif

Writing recursive_sum.cpp


In [2]:
!g++ -std=c++17 -O2 -o recursive_sum recursive_sum.cpp && ./recursive_sum

a = [3, 1, 4, 1, 5, 9, 2, 6]
RecursiveSum(a, 8) = 31 with 8 calls
sumRecDecrease(a, 8) = 31 with 9 calls
n  measured S(n)  3n-1
1             2     2
2             5     5
3             8     8
4            11    11
5            14    14
6            17    17
7            20    20
8            23    23


**Try it yourself.** Change the array `a` (also its length) and check that the measured steps are always $3n-1$.

In [3]:
%%writefile prova_recursive_sum.cpp
#define NOMAIN
#include "recursive_sum.cpp"

int main()
{
    // Change the array and check that the measured steps are always 3n-1.
    int a[] = {7, -2, 5, 0, 11};
    int n = 5;
    steps = 0;
    int r = RecursiveSum(a, n);
    printf("%d %lld %d\n", r, steps, 3 * n - 1);
    return 0;
}

Writing prova_recursive_sum.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_recursive_sum prova_recursive_sum.cpp && ./prova_recursive_sum

21 14 14


## 2. Tower of Hanoi: $T(n)=2T(n-1)+1$

**Hanoi**$(n, from, to, aux)$ — *Input*: the number of disks $n\ge1$ and the three pegs. *Output*: the sequence of moves that transfers disks $1,\dots,n$ from *from* to *to*.

The code counts the moves (and, for $n=3$, prints them): the chapter shows that they are exactly $2^n-1$. The initial call is `Hanoi(n, 'A', 'C', 'B')`.

In [5]:
%%writefile hanoi.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long moves = 0;    // number of moves
bool printMoves = false;

void Hanoi(int n, char from, char to, char aux)
{
    if (n == 1)
    {
        moves++;                                // move disk 1
        if (printMoves) printf("  disk 1: %c -> %c\n", from, to);
        return;
    }
    Hanoi(n - 1, from, aux, to);
    moves++;                                    // move disk n
    if (printMoves) printf("  disk %d: %c -> %c\n", n, from, to);
    Hanoi(n - 1, aux, to, from);
}

#ifndef NOMAIN
int main()
{
    printf("Moves for n = 3:\n");
    printMoves = true;
    Hanoi(3, 'A', 'C', 'B');
    printMoves = false;
    printf("n   moves  2^n-1\n");
    int ns[] = {1, 2, 3, 4, 5, 6, 7, 8, 10};
    for (int n : ns)
    {
        moves = 0;
        Hanoi(n, 'A', 'C', 'B');
        printf("%-3d %5lld %5lld\n", n, moves, (1LL << n) - 1);
    }
    return 0;
}
#endif

Writing hanoi.cpp


In [6]:
!g++ -std=c++17 -O2 -o hanoi hanoi.cpp && ./hanoi

Moves for n = 3:
  disk 1: A -> C
  disk 2: A -> B
  disk 1: C -> B
  disk 3: A -> C
  disk 1: B -> A
  disk 2: B -> C
  disk 1: A -> C
n   moves  2^n-1
1       1     1
2       3     3
3       7     7
4      15    15
5      31    31
6      63    63
7     127   127
8     255   255
10   1023  1023


**Try it yourself.** Change the number of disks $n$: the moves grow as $2^n-1$ (with $n=15$ they are already $32\,767$).

In [7]:
%%writefile prova_hanoi.cpp
#define NOMAIN
#include "hanoi.cpp"

int main()
{
    // Try with more disks: how many moves are needed for n = 15?
    int n = 15;
    moves = 0;
    Hanoi(n, 'A', 'C', 'B');
    printf("%d %lld %lld\n", n, moves, (1LL << n) - 1);
    return 0;
}

Writing prova_hanoi.cpp


In [8]:
!g++ -std=c++17 -O2 -o prova_hanoi prova_hanoi.cpp && ./prova_hanoi

15 32767 32767


## 3. Recurrences computed directly and closed forms

Every recurrence is computed by a recursive function that follows the definition and compared with the closed form obtained in the chapter (expansion method). For the recurrences with $n/2$ the values of $n$ are powers of 2.

| recurrence | closed form |
|---|---|
| $T(n)=T(n-1)+n$, $T(1)=1$ | $n(n+1)/2$ |
| $S(n)=S(n-1)+3$, $S(1)=2$ (RecursiveSum) | $3n-1$ |
| $T(n)=2T(n-1)+1$, $T(1)=1$ (Hanoi) | $2^n-1$ |
| $T(n)=T(n/2)+1$, $T(1)=1$ | $\log_2 n+1$ |
| $T(n)=2T(n/2)+n$, $T(1)=1$ | $n\log_2 n+n$ |

In [9]:
%%writefile recurrences.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long TQuad(long long n)  { return n == 1 ? 1 : TQuad(n - 1) + n; }
long long TSum(long long n)   { return n == 1 ? 2 : TSum(n - 1) + 3; }
long long THanoi(long long n) { return n == 1 ? 1 : 2 * THanoi(n - 1) + 1; }
long long TLog(long long n)   { return n == 1 ? 1 : TLog(n / 2) + 1; }
long long TNLogN(long long n) { return n == 1 ? 1 : 2 * TNLogN(n / 2) + n; }

long long Log2(long long n)   // n is a power of 2
{
    return 63 - __builtin_clzll(n);
}

void Confronta(const string &nome, vector<long long> valori,
               function<long long(long long)> ric, function<long long(long long)> chiusa)
{
    printf("%s\n", nome.c_str());
    for (long long n : valori)
    {
        long long r = ric(n), c = chiusa(n);
        printf("  n=%-7lld %12lld %12lld  %s\n", n, r, c, r == c ? "OK" : "DIFFERENT");
    }
}

#ifndef NOMAIN
int main()
{
    Confronta("T(n) = T(n-1) + n  ->  n(n+1)/2", {1, 2, 3, 4, 10, 100}, TQuad, [](long long n) { return n * (n + 1) / 2; });
    Confronta("S(n) = S(n-1) + 3  ->  3n-1", {1, 2, 3, 4, 100}, TSum, [](long long n) { return 3 * n - 1; });
    Confronta("T(n) = 2T(n-1) + 1  ->  2^n-1", {1, 2, 3, 4, 10, 20}, THanoi, [](long long n) { return (1LL << n) - 1; });
    Confronta("T(n) = T(n/2) + 1  ->  log2(n)+1", {1, 2, 4, 8, 1024, 1 << 20}, TLog, [](long long n) { return Log2(n) + 1; });
    Confronta("T(n) = 2T(n/2) + n  ->  n log2(n) + n", {1, 2, 4, 8, 1024, 1 << 20}, TNLogN, [](long long n) { return n * Log2(n) + n; });
    return 0;
}
#endif

Writing recurrences.cpp


In [10]:
!g++ -std=c++17 -O2 -o recurrences recurrences.cpp && ./recurrences

T(n) = T(n-1) + n  ->  n(n+1)/2
  n=1                  1            1  OK
  n=2                  3            3  OK
  n=3                  6            6  OK
  n=4                 10           10  OK
  n=10                55           55  OK
  n=100             5050         5050  OK
S(n) = S(n-1) + 3  ->  3n-1
  n=1                  2            2  OK
  n=2                  5            5  OK
  n=3                  8            8  OK
  n=4                 11           11  OK
  n=100              299          299  OK
T(n) = 2T(n-1) + 1  ->  2^n-1
  n=1                  1            1  OK
  n=2                  3            3  OK
  n=3                  7            7  OK
  n=4                 15           15  OK
  n=10              1023         1023  OK
  n=20           1048575      1048575  OK
T(n) = T(n/2) + 1  ->  log2(n)+1
  n=1                  1            1  OK
  n=2                  2            2  OK
  n=4                  3            3  OK
  n=8                  4            4

**Try it yourself.** Add a recurrence of your own (here $T(n)=T(n-1)+2n-1$, $T(1)=1$) and compare it with its closed form.

In [11]:
%%writefile prova_recurrences.cpp
#define NOMAIN
#include "recurrences.cpp"

int main()
{
    // Add a recurrence: T(n) = T(n-1) + 2n - 1 with T(1) = 1 has closed form n^2.
    function<long long(long long)> tMia = [&](long long n) -> long long { return n == 1 ? 1 : tMia(n - 1) + 2 * n - 1; };
    Confronta("T(n) = T(n-1) + 2n - 1  ->  n^2", {1, 2, 3, 10, 100}, tMia, [](long long n) { return n * n; });
    return 0;
}

Writing prova_recurrences.cpp


In [12]:
!g++ -std=c++17 -O2 -o prova_recurrences prova_recurrences.cpp && ./prova_recurrences

T(n) = T(n-1) + 2n - 1  ->  n^2
  n=1                  1            1  OK
  n=2                  4            4  OK
  n=3                  9            9  OK
  n=10               100          100  OK
  n=100            10000        10000  OK
